# Verified Phase 2 lineage — Natural Sampling Phase 1

This notebook belongs to the corrected AOI-masked Phase 2 workflow. It must use only:

- `Source/Project/final_selected_phase2_models.json` as the immutable Phase 2 checkpoint registry;
- `Results/Final_Article_Harmonized_GEDIAnchored_NaturalP1` for evaluation products;
- `Inference_Harmonized_GEDIAnchored_NaturalP1` for annual maps.

The former `Phase2_AOI_Masked`, `Final_Article_AOI_Masked`, and `Inference_AOI_Masked` products were generated from an incorrect Phase 1 parent lineage and must not be used. Run `Phase_2.ipynb` first, followed by `Inference.ipynb`, before regenerating downstream figures.


> **Active final lineage.** All three landscapes use the selected residual-only Phase 2 checkpoints. Exact paths and SHA-256 hashes are frozen in `Source/Project/final_selected_phase2_models.json`. Execute `Phase_2.ipynb`, `Inference.ipynb`, and `Chm_Comparison.ipynb` first.


# Natural Sampling publication pipeline

This notebook is the isolated Natural Sampling copy. The original official pipeline remains unchanged. 
Training is disabled because the selected Phase 1 and Phase 2 models are already archived under `Natural_Sampling/Models`. 
Run the notebook from the first cell to regenerate figures and tables under `Natural_Sampling/Results`.


> Dependency: run `Inference.ipynb`, then `Chm_Comparison.ipynb`, before executing this notebook. This Natural Sampling copy never falls back to results from the former official model.


# MSE decomposition — three Moroccan forest ecosystems

This notebook reproduces the additive MSE decomposition used by Schwartz et al.:

\[
\mathrm{MSE}=\mathrm{SB}+\mathrm{SDSD}+\mathrm{LCS},
\]

with:

- **SB** (squared bias): \((\bar{p}-\bar{o})^2\);
- **SDSD** (difference in standard deviations): \((s_p-s_o)^2\);
- **LCS** (lack of correlation weighted by dispersion): \(2s_ps_o(1-r)\).

All products within one forest are evaluated on the **strict intersection of identical GEDI test shots**. This prevents coverage differences from favouring any product. Potapov is omitted for Agadir because no valid product support is available there.


## 1. Imports and immutable paths

The point-level cache was produced by the verified CHM comparison workflow. It contains the GEDI shot identifier, observed RH95, prediction, coverage, product, and forest.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
CACHE = ROOT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "CHM_Comparison" / "GEDI_TEST_product_valid_support_signed_errors.csv.gz"
OUT = ROOT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "MSE_Decomposition"
OUT.mkdir(parents=True, exist_ok=True)
if not CACHE.is_file():
    raise FileNotFoundError(
        "Natural Sampling comparison cache is missing:\n"
        f"  {CACHE}\n\n"
        "Required execution order:\n"
        "  1. Run Inference.ipynb from the first cell to generate the Natural Sampling Phase-2 maps.\n"
        "  2. Run Chm_Comparison.ipynb from the first cell to create the paired GEDI/product cache.\n"
        "  3. Re-run this MSE_Decomposition.ipynb notebook.\n\n"
        "The official-pipeline cache is intentionally not reused because it was generated "
        "from different model maps."
    )

FORESTS = ["Ifran", "Maamoura", "Agadir"]
PRODUCTS = {
    "Our B4 Phase 2": "Our Model",
    "Pauls 2020": "Pauls et al. (Pa24)",
    "Lang 2020": "Lang et al. (L23)",
    "Meta/Tolan 2023": "Tolan et al. (T24)",
    "GFCH 2019": "Potapov et al. (P21)",
}
PRODUCT_ORDER = list(PRODUCTS.values())
print("Cache:", CACHE)
print("Output:", OUT)


## 2. Strict-common support

For each forest, a shot is retained only when RH95 and predictions are finite for **all products available in that forest**. The cell asserts that every product has the same ordered shot IDs and the same GEDI RH95 values.


In [ ]:
raw = pd.read_csv(CACHE)
required = {"forest", "product", "shot_id", "rh95", "prediction"}
assert required.issubset(raw.columns), sorted(required - set(raw.columns))
raw = raw[list(required)].copy()
raw = raw[np.isfinite(raw["rh95"]) & np.isfinite(raw["prediction"])]
raw["display_product"] = raw["product"].map(PRODUCTS)
raw = raw[raw["display_product"].notna()].copy()

strict_by_forest = {}
support_rows = []
for forest in FORESTS:
    part = raw[raw["forest"].eq(forest)].copy()
    available = [p for p in PRODUCT_ORDER if p in set(part["display_product"])]
    if forest == "Agadir":
        assert "Potapov et al. (P21)" not in available
    wide = part.pivot_table(index=["shot_id", "rh95"], columns="display_product", values="prediction", aggfunc="first")
    wide = wide.dropna(subset=available).reset_index().sort_values("shot_id").reset_index(drop=True)
    assert len(wide) > 0, forest
    strict_by_forest[forest] = wide
    for product in available:
        support_rows.append({"Forest": forest, "Product": product, "n": len(wide)})
    assert len({row["n"] for row in support_rows if row["Forest"] == forest}) == 1

support = pd.DataFrame(support_rows)
display(support)
support.to_csv(OUT / "strict_common_support.csv", index=False)


## 3. Additive decomposition and numerical guards

Population standard deviations (`ddof=0`) are used so the algebraic equality is exact for the empirical MSE. The residual between direct MSE and the three-term sum must remain below numerical tolerance.


In [ ]:
def decomposition(obs, pred):
    obs = np.asarray(obs, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)
    assert obs.shape == pred.shape and obs.size > 1
    error = pred - obs
    mse = float(np.mean(error**2))
    rmse = float(np.sqrt(mse))
    sb = float((pred.mean() - obs.mean())**2)
    so, sp = float(obs.std(ddof=0)), float(pred.std(ddof=0))
    corr = float(np.corrcoef(obs, pred)[0, 1])
    sdsd = float((sp - so)**2)
    lcs = float(2.0 * sp * so * (1.0 - corr))
    reconstructed = sb + sdsd + lcs
    assert np.isclose(mse, reconstructed, rtol=1e-9, atol=1e-9), (mse, reconstructed)
    ss_res = float(np.sum(error**2))
    ss_tot = float(np.sum((obs - obs.mean())**2))
    r2 = 1.0 - ss_res / ss_tot
    return {"n": obs.size, "SB": sb, "SDSD": sdsd, "LCS": lcs,
            "MSE": mse, "RMSE": rmse, "R2": r2, "Corr": corr,
            "closure_error": mse - reconstructed}

rows = []
for forest, wide in strict_by_forest.items():
    for product in PRODUCT_ORDER:
        if product not in wide.columns:
            continue
        rows.append({"Forest": forest, "Method": product,
                     **decomposition(wide["rh95"], wide[product])})

metrics = pd.DataFrame(rows)
metrics["order"] = metrics["Method"].map({p: i for i, p in enumerate(PRODUCT_ORDER)})
metrics = metrics.sort_values(["Forest", "order"]).reset_index(drop=True)
assert metrics.groupby("Forest")["n"].nunique().eq(1).all()
assert metrics["closure_error"].abs().max() < 1e-8
display(metrics.drop(columns="order"))
metrics.to_csv(OUT / "MSE_decomposition_strict_common_metrics.csv", index=False)


## 4. Article figure

Each ecological domain is a separate panel. Bars are ordered as **Our Model, Pauls, Lang, Tolan, Potapov**; unavailable products are omitted. The table below each panel reports RMSE and coefficient of determination on the same strict-common support.


In [ ]:
# STEP_LATEX_CLEAN_COLORED_MSE_DECOMPOSITION_V1
# Textual headings, axis names and method labels are intentionally delegated
# to LaTeX. Numerical ticks and metric values remain part of the scientific
# graphic so that the scale is never ambiguous.
COLORS = {
    "SB": "#4C78A8",      # systematic bias
    "SDSD": "#F28E2B",    # difference in standard deviations
    "LCS": "#59A14F",     # lack of correlation
}
METHOD_SHORT = {
    "Our Model": "Our model",
    "Pauls et al. (Pa24)": "Pa24",
    "Lang et al. (L23)": "L23",
    "Tolan et al. (T24)": "T24",
    "Potapov et al. (P21)": "P21",
}

global_max = float(metrics["MSE"].max())
ymax = math.ceil((global_max * 1.10) / 10.0) * 10.0


def draw_mse_panel(ax, forest):
    part = metrics[metrics["Forest"].eq(forest)].sort_values("order")
    x = np.arange(len(part))
    bottom = np.zeros(len(part))
    for component in ("SB", "SDSD", "LCS"):
        ax.bar(
            x, part[component], bottom=bottom, width=0.72,
            color=COLORS[component], edgecolor="white", linewidth=0.7,
        )
        bottom += part[component].to_numpy()
    ax.set_ylim(0, ymax)
    ax.set_xticks(x)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    # Metric rows are exported separately and composed as native LaTeX.
    return part


# Separate vector panels give LaTeX full control over ecosystem headings,
# model labels, spacing and the shared legend.
for forest in FORESTS:
    fig, ax = plt.subplots(figsize=(5.1, 5.2))
    draw_mse_panel(ax, forest)
    fig.subplots_adjust(left=0.10, right=0.995, top=0.995, bottom=0.27)
    for suffix in ("png", "svg", "pdf"):
        path = OUT / f"MSE_decomposition_{forest}_strict_common_LATEX_CLEAN.{suffix}"
        fig.savefig(
            path,
            dpi=900 if suffix == "png" else None,
            bbox_inches="tight",
            facecolor="white",
        )
        print("Saved:", path)
    plt.show()
    plt.close(fig)


# A compact combined fallback is also exported.  Each ecosystem panel
# carries its own internal component legend for standalone readability.
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(14.8, 5.2), sharey=True)
for ax, forest in zip(axes, FORESTS):
    draw_mse_panel(ax, forest)

legend_handles = [
    Patch(facecolor=COLORS[name], edgecolor="white", linewidth=0.7, label=name)
    for name in ("SB", "SDSD", "LCS")
]
for ax in axes:
    ax.legend(
        handles=legend_handles,
        loc="upper left",
        ncol=1,
        frameon=True,
        framealpha=0.86,
        edgecolor="0.75",
        fontsize=10,
        handlelength=1.7,
        borderpad=0.6,
        labelspacing=0.45,
    )
fig.subplots_adjust(left=0.045, right=0.998, top=0.995, bottom=0.10, wspace=0.08)
for suffix in ("png", "svg", "pdf"):
    path = OUT / f"MSE_decomposition_three_forests_strict_common_LATEX_CLEAN.{suffix}"
    fig.savefig(
        path,
        dpi=900 if suffix == "png" else None,
        bbox_inches="tight",
        pad_inches=0.08,  # keep titles clear of the upper axis spine
        facecolor="white",
    )
    print("Saved:", path)
plt.show()
plt.close(fig)


## 5. Interpretation export

The dominant component identifies the main error mechanism:

- **SB-dominated:** systematic mean bias;
- **SDSD-dominated:** predicted height variability differs from GEDI;
- **LCS-dominated:** insufficient pointwise correspondence despite similar mean and spread.

The following table reports component percentages for direct use in the Results and Discussion sections.


In [ ]:
percentages = metrics[["Forest", "Method", "n", "MSE", "RMSE", "R2", "Corr"]].copy()
for component in ("SB", "SDSD", "LCS"):
    percentages[f"{component}_pct"] = 100.0 * metrics[component] / metrics["MSE"]
display(percentages)
percentages.to_csv(OUT / "MSE_decomposition_component_percentages.csv", index=False)
print("All numerical and graphical outputs are ready in:", OUT)


## Final support guard — MSE decomposition

The MSE decomposition is computed on the same strict-common `shot_id`
intersection as the CHM benchmark. For Agadir, Potapov/P21 is not included
because its valid spatial coverage is insufficient. The remaining methods
(Our Model, Pauls, Lang and Tolan) must each contain the identical 6,401-shot
support before SB, SDSD and LCS are calculated.


In [ ]:
# AGADIR_MSE_STRICT_COMMON_GUARD_V1
_agadir_wide = strict_by_forest["Agadir"]
_agadir_methods = [name for name in PRODUCT_ORDER if name in _agadir_wide.columns]
assert "Potapov et al. (P21)" not in _agadir_methods
assert len(_agadir_wide) == 6401, len(_agadir_wide)
assert not _agadir_wide[_agadir_methods].isna().any().any()
assert _agadir_wide["shot_id"].astype(str).nunique() == 6401
assert metrics.loc[metrics["Forest"].eq("Agadir"), "n"].nunique() == 1
assert int(metrics.loc[metrics["Forest"].eq("Agadir"), "n"].iloc[0]) == 6401
display(pd.DataFrame({
    "forest": ["Agadir"] * len(_agadir_methods),
    "method": _agadir_methods,
    "identical_strict_common_n": [6401] * len(_agadir_methods),
}))
print("[PASS] Agadir MSE decomposition uses identical strict-common shot IDs (n=6,401).")
